# pandas: a demographic table for a policy question — worked solutions

Load, inspect, filter, transform, and summarise demographic data for a policy question.

Run cells in order with **Shift + Enter**. Each notebook needs its own imports and definitions.


## Data and research context

`data/wdi_population.csv` contains eight countries at four selected years: 2010, 2015, 2020, and 2023.
One row is a **country-year**. This is a fixed extract of published World Bank indicators.
`population` is a count; `share_65_plus` and `urban_share` are percentages; `life_expectancy` is
life expectancy **at birth**, in years, not the average age of people who died.

These countries are not a representative sample of the world. Country-level patterns do not establish
individual-level relationships or causal effects. Derived older population counts are approximate.
See `data/DATA_NOTE_EN.txt` for definitions and provenance.

Sources: [population](https://data.worldbank.org/indicator/SP.POP.TOTL),
[65+ share](https://data.worldbank.org/indicator/SP.POP.65UP.TO.ZS),
[life expectancy](https://data.worldbank.org/indicator/SP.DYN.LE00.IN),
[urban share](https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS).


## PD-01 · Load a real demographic table

Import pandas as pd and read data/wdi_population.csv into df.
Display its first five rows, its shape, and its column information.
Identify the observational unit and the variables' units.

The file contains World Bank indicators for eight countries in 2010, 2015, 2020, and 2023.
One row is one country in one year. Values are a saved historical extract, not forecasts.
See data/DATA_NOTE_EN.txt for definitions and sources. The file is already local.


In [1]:
import pandas as pd

df = pd.read_csv("data/wdi_population.csv")
print("Rows and columns:", df.shape)
df.head()

Rows and columns: (32, 7)


,country_code,country,year,population,share_65_plus,life_expectancy,urban_share
0,BRA,Brazil,2010,1.937019e+08,6.666886,73.779,84.343846
1,BRA,Brazil,2015,2.016755e+08,7.956807,75.106,85.693570
2,BRA,Brazil,2020,2.086608e+08,9.598665,74.506,86.911219
3,BRA,Brazil,2023,2.111407e+08,10.633631,75.848,87.615971
4,CHN,China,2010,1.337705e+09,8.661471,75.668,49.234027


## PD-01b · Inspect column information

Call info on df. Check that year and the demographic indicators are numeric.
Text columns may be labelled str, string, or object depending on the pandas version.


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country_code     32 non-null     str    
 1   country          32 non-null     str    
 2   year             32 non-null     int64  
 3   population       32 non-null     float64
 4   share_65_plus    32 non-null     float64
 5   life_expectancy  32 non-null     float64
 6   urban_share      32 non-null     float64
dtypes: float64(4), int64(1), str(2)
memory usage: 1.9 KB


Each row describes one country in one year. Population is measured in people, the two shares in percent, and life expectancy at birth in years. These indicators and the calendar year all have numeric types.

## PD-02 · Choose a year, select columns, and sort

Create snapshot as a copy of the rows for 2023. Display its shape.
Select country, population, and share_65_plus, then sort the table by share_65_plus in descending order.
Explain why this comparison uses one year rather than mixing observations from different years.


In [3]:
snapshot = df.loc[df["year"] == 2023].copy()
print("Rows and columns in 2023:", snapshot.shape)
snapshot[["country", "population", "share_65_plus"]].sort_values(
    "share_65_plus", ascending=False
)

Rows and columns in 2023: (8, 7)


,country,population,share_65_plus
19,Japan,1.245166e+08,29.561810
11,Germany,8.328727e+07,22.790044
27,Russian Federation,1.438261e+08,16.604643
7,China,1.410710e+09,14.316077
3,Brazil,2.111407e+08,10.633631
23,Mexico,1.297398e+08,7.986646
15,India,1.438070e+09,6.921843
31,South Africa,6.321238e+07,6.515094


Filtering to 2023 gives every country the same reference year and one row in the ranking.

## PD-03 · Check missingness and create a useful variable

Count missing values in each column. Check for duplicate country_code–year keys.
In snapshot, estimate the number of people aged 65+ as population × share_65_plus / 100.
Name the new column estimated_65_plus. Display country and this estimate.

Missing is not zero. Do not invent missing values, fill them with zero, or drop rows automatically.
The estimate combines published indicators and is approximate, not a separate count from a census.


In [4]:
print("Missing values by column:")
print(df.isna().sum())
print("Duplicate country-year keys:", df.duplicated(["country_code", "year"]).sum())

snapshot["estimated_65_plus"] = snapshot["population"] * snapshot["share_65_plus"] / 100
snapshot[["country", "estimated_65_plus"]].round({"estimated_65_plus": 0})

Missing values by column:
country_code       0
country            0
year               0
population         0
share_65_plus      0
life_expectancy    0
urban_share        0
dtype: int64
Duplicate country-year keys: 0


,country,estimated_65_plus
3,Brazil,22451927.0
7,China,201958336.0
11,Germany,18981206.0
15,India,99540924.0
19,Japan,36809376.0
23,Mexico,10361855.0
27,Russian Federation,23881815.0
31,South Africa,4118346.0


## PD-04 · Group observations and summarise

Group df by year and calculate total population across the eight selected countries.
Then report the number of country observations in each year.
Explain why the totals are not world population and why summing all years would not be a valid
single-year population total.


In [5]:
df.groupby("year").agg(
    total_population=("population", "sum"),
    country_observations=("country_code", "count"),
)

,total_population,country_observations
year,,
2010,3.293553e+09,8
2015,3.440824e+09,8
2020,3.564407e+09,8
2023,3.604503e+09,8


Each total covers the eight countries in the extract. Adding the four totals would count many of the same residents in multiple years.

## PD-05 · Prepare a small policy briefing table

Using the 2023 snapshot, keep countries whose share aged 65+ is at least 20%.
Display country, share_65_plus, and estimated_65_plus; sort by the older population share.
Write two sentences: one describing your result and one explaining what additional information
a public administrator would need before allocating service resources.


In [6]:
columns = ["country", "share_65_plus", "estimated_65_plus"]
briefing = snapshot.loc[snapshot["share_65_plus"] >= 20, columns].sort_values(
    "share_65_plus", ascending=False
)
briefing.round({"share_65_plus": 1, "estimated_65_plus": 0})

,country,share_65_plus,estimated_65_plus
19,Japan,29.6,36809376.0
11,Germany,22.8,18981206.0


In 2023, Japan (29.6%) and Germany (22.8%) met the threshold among the eight countries. Before allocating services, a planner would also need local estimates of unmet care needs and available service capacity.

# Additional exercises


## PD-X1 · Missing values and income categories in Adult

Read data/adult.data.csv, recognising ? as missing. Count missing values.
Create high_income by mapping <=50K to 0 and >50K to 1 in the salary column.
Group by education and report the mean and count of this indicator.
The data are historical person records; summaries here are unweighted and descriptive.
The field salary is an income category, not an exact salary amount.


In [7]:
adult = pd.read_csv("data/adult.data.csv", na_values="?")
print("Missing values by column:")
print(adult.isna().sum())

adult["high_income"] = adult["salary"].map({"<=50K": 0, ">50K": 1})
adult.groupby("education")["high_income"].agg(["mean", "count"])

Missing values by column:
age                  0
workclass         1836
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     583
salary               0
dtype: int64


,mean,count
education,,
10th,0.066452,933
11th,0.051064,1175
12th,0.076212,433
1st-4th,0.035714,168
5th-6th,0.048048,333
7th-8th,0.061920,646
9th,0.052529,514
Assoc-acdm,0.248360,1067
Assoc-voc,0.261216,1382


Within each education group, the mean of the 0/1 indicator is the fraction of records with income above 50K. The count gives the number of records with a known income category in that group.

## PD-X2 · Export a summary and use two conditions

Select country and year for records from 2023 with an older population share of at least 20%.
Use .loc and two parenthesised conditions joined by &.
Save your briefing table as outputs/ageing_briefing.csv, without an extra numeric index.
The file should contain the results and meaningful column names. Rerunning replaces that output file.


In [8]:
from pathlib import Path

selected = df.loc[(df["year"] == 2023) & (df["share_65_plus"] >= 20), ["country", "year"]]
print(selected.to_string(index=False))

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
briefing.to_csv(output_dir / "ageing_briefing.csv", index=False)

country  year
Germany  2023
  Japan  2023


## Save and check

Save the completed notebook, restart the kernel, and run from the top. Keep the data folder beside the notebooks.
